# Quickstart 1 — forward models and simulation

One capability per cell, on synthetic data, nothing hidden. This notebook shows
the **forward models** (what an epoch radial velocity and an along-scan
abscissa *are*, as functions of orbital elements) and the **simulators** that
generate demo data from a truth.

Conventions used throughout (`docs/model_and_likelihoods.md`, §1): epochs in TCB MJD;
angles in radians; the scan angle ψ counter-clockwise from north; a Keplerian
year is `DAYS_PER_KEPLER_YEAR` days; ω is the primary-frame argument of
periastron; the astrometric forward models take the primary's orbit as the
photocentre (**β = 0**, a dark companion).

The open-surface contract these names belong to: `docs/open_api.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from orblet.constants import DAYS_PER_KEPLER_YEAR, MJD_J2010_TCB
from orblet.simulate.bundles import load_simulated_inputs
from orblet import prepare_rv_for_orbit, resolve_epochs_mjd, semi_amplitude_kms

# The toy-orbit truth on an invented sky position, with loader-shaped RV and
# along-scan data on a parallax-consistent 5-year cadence.
bundle = load_simulated_inputs(seed=1)
truth = bundle.truth
astro = bundle.astro_data
prepared = prepare_rv_for_orbit(bundle.rv_data, time_scale="gaia_obmt")

EPOCH_REF = float(truth.t_ref_mjd)                      # the fit's reference epoch (MJD)
P_YR = truth.P_days / DAYS_PER_KEPLER_YEAR              # forward models take Keplerian years
TAU = ((truth.tp_mjd - EPOCH_REF) / truth.P_days) % 1.0 # periastron phase in [0, 1)
print(f"P = {truth.P_days:.1f} d, e = {truth.e:.2f}, i = {np.degrees(truth.i_rad):.1f} deg, "
      f"m1 = {truth.m1_msun:.2f}, m2 = {truth.m2_msun:.2f} Msun, plx = {truth.parallax_mas:.2f} mas")

## The RV forward model

`rv_model` returns the primary's radial velocity. Its `mass_msun` argument is
the **projected** companion mass m₂ sin i (the RV channel never sees i on its
own); `semi_amplitude_kms` is the edge-on semi-amplitude for the same masses.

In [ ]:
from orblet.model import rv_model

# Draw the curve over the span of the data. The reference epoch sits AFTER the
# last observation (the cadence runs 2010-2015, the epoch is J2016), so a window
# around EPOCH_REF would show a curve where there are no points to compare it to.
t_rv = prepared["epochs_mjd"]
t_grid = np.linspace(t_rv.min() - 0.25 * truth.P_days, t_rv.max() + 0.25 * truth.P_days, 1200)
rv_curve_kms = rv_model(
    t_grid, period_yr=P_YR, ecc=truth.e, omega_rad=truth.omega_rad, tau=TAU,
    mass_msun=truth.m2_msun * np.sin(truth.i_rad), M_msun=truth.M_total_msun,
    offset_kms=truth.gamma_kms, epoch_ref_mjd=EPOCH_REF,
)
K_edge_on = semi_amplitude_kms(mass_msun=truth.m2_msun, period_yr=P_YR, ecc=truth.e,
                               M_total_msun=truth.M_total_msun)
print(f"K edge-on = {K_edge_on:.2f} km/s; x sin i = {K_edge_on * np.sin(truth.i_rad):.2f} km/s; "
      f"simulator K1 = {truth.K1_kms:.2f} km/s")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t_grid - EPOCH_REF, rv_curve_kms, lw=1)
ax.errorbar(prepared["epochs_mjd"] - EPOCH_REF, prepared["rv"], prepared["rv_err"], fmt=".", ms=4)
ax.set_xlabel("days from reference epoch"); ax.set_ylabel("RV (km/s)")
plt.show()

In [ ]:
from orblet.likelihood import rv_loglike

# The common Gaussian-with-jitter likelihood; write your own when you need
# another (Student-t, outlier mixture, ...): the forward model is the primitive.
def ll_rv(period_yr):
    return rv_loglike(
        prepared["epochs_mjd"], prepared["rv"], prepared["rv_err"],
        period_yr=period_yr, ecc=truth.e, omega_rad=truth.omega_rad, tau=TAU,
        mass_msun=truth.m2_msun * np.sin(truth.i_rad), M_msun=truth.M_total_msun,
        offset_kms=truth.gamma_kms, jitter_kms=0.0, epoch_ref_mjd=EPOCH_REF,
    )
print(f"log L at the truth: {ll_rv(P_YR):.1f};  at 1.3 x P: {ll_rv(1.3 * P_YR):.1f}")

## The astrometric forward models

Two parametrisations of the same photocentre orbit: **Campbell** elements with
masses (`campbell_xy`, which assumes a dark companion so the primary's orbit is
the photocentre) and **Thiele–Innes** amplitudes A, B, F, G in mas
(`thiele_innes_xy`). Building A, B, F, G by hand from the Campbell elements and
checking the two agree is the cheapest convention test there is.

In [ ]:
from orblet.model import campbell_xy, thiele_innes_xy, kepler_xy_orbit, tp_from_disk_angle

t_mjd = resolve_epochs_mjd(astro)          # loader obs_time (J2010-days) -> TCB MJD
psi = np.asarray(astro["scan_angle"], dtype=float)
pf = np.asarray(astro["parallax_factor_al"], dtype=float)

d_ra_c, d_dec_c = campbell_xy(
    t_mjd, period_yr=P_YR, ecc=truth.e, omega=truth.omega_rad, inc=truth.i_rad,
    Omega=truth.Omega_rad, tp_mjd=truth.tp_mjd, m_comp_msun=truth.m2_msun,
    M_total_msun=truth.M_total_msun, plx_mas=truth.parallax_mas,
)

# Thiele-Innes constants from the Campbell elements (Hilditch 2001, section 5.12),
# with the photocentre amplitude a_phot = (m2 / M_total) * a * plx in mas.
a, w, W, ci = truth.a_phot_mas, truth.omega_rad, truth.Omega_rad, np.cos(truth.i_rad)
A = a * ( np.cos(w) * np.cos(W) - np.sin(w) * np.sin(W) * ci)
B = a * ( np.cos(w) * np.sin(W) + np.sin(w) * np.cos(W) * ci)
F = a * (-np.sin(w) * np.cos(W) - np.cos(w) * np.sin(W) * ci)
G = a * (-np.sin(w) * np.sin(W) + np.cos(w) * np.cos(W) * ci)
d_ra_t, d_dec_t = thiele_innes_xy(t_mjd, period_yr=P_YR, ecc=truth.e,
                                  A_mas=A, B_mas=B, F_mas=F, G_mas=G, tp_mjd=truth.tp_mjd)
print("max |Campbell - Thiele-Innes| (mas):", np.max(np.abs(d_ra_c - d_ra_t)), np.max(np.abs(d_dec_c - d_dec_t)))

# The in-plane track (normalised by a) and the periastron-time convention helper.
x_orb, y_orb = kepler_xy_orbit(t_mjd, period_yr=P_YR, ecc=truth.e, tp_mjd=truth.tp_mjd)
tp_back = tp_from_disk_angle(np.cos(2 * np.pi * TAU), np.sin(2 * np.pi * TAU),
                             period_days=truth.P_days, epoch_ref_mjd=EPOCH_REF)
print(f"tp from the phase latent: {tp_back:.3f}; truth tp: {truth.tp_mjd:.3f} (equal modulo P)")

## Along-scan projection: what Gaia measures

One number per transit: the sky offset projected onto the scan direction, plus
the five-parameter single-star model (position offset, proper motion, parallax
times Gaia's own along-scan parallax factor). `along_scan_model` is that
projection; `loglike_along_scan` the Gaussian likelihood; `along_scan_from_theta`
recomputes the engine's model from a flat parameter dict.

In [ ]:
from orblet.model import along_scan_model, along_scan_from_theta
from orblet.likelihood import loglike_along_scan

model_al = along_scan_model(
    d_ra=d_ra_c, d_dec=d_dec_c, psi=psi, t_mjd=t_mjd, epoch_ref_mjd=EPOCH_REF,
    ra_offset_mas=0.0, dec_offset_mas=0.0,
    pmra_masyr=truth.pmra_masyr, pmdec_masyr=truth.pmdec_masyr,
    plx_mas=truth.parallax_mas, parallax_factor_al=pf,
)
d_obs = np.asarray(astro["centroid_pos"], dtype=float)
sigma = np.asarray(astro["centroid_pos_err"], dtype=float)
resid = d_obs - model_al
print(f"rms residual at the truth: {resid.std():.3f} mas; per-epoch sigma: {np.median(sigma):.3f} mas")
print("log L (along-scan):", loglike_along_scan(model_along_scan=model_al, centroid_pos=d_obs,
                                               centroid_pos_err=sigma, jitter_mas=0.0))

# The same model from a flat theta dict in the Thiele-Innes basis (P in DAYS here;
# the periastron enters through the unit-disk phase latents).
theta_ti = {
    "P_days": truth.P_days, "e": truth.e,
    "A_mas": A, "B_mas": B, "F_mas": F, "G_mas": G,
    "phase_x": np.cos(2 * np.pi * TAU), "phase_y": np.sin(2 * np.pi * TAU),
    "ra_offset_mas": 0.0, "dec_offset_mas": 0.0,
    "pmra_masyr": truth.pmra_masyr, "pmdec_masyr": truth.pmdec_masyr,
    "plx_mas": truth.parallax_mas,
}
model_theta = along_scan_from_theta(theta_ti, basis="thiele_innes", t_mjd=t_mjd, psi=psi,
                                    parallax_factor_al=pf, epoch_ref_mjd=EPOCH_REF)
print("max |along_scan_model - along_scan_from_theta| (mas):", np.max(np.abs(model_al - model_theta)))

fig, ax = plt.subplots(figsize=(7, 3))
ax.errorbar(t_mjd - EPOCH_REF, d_obs, sigma, fmt=".", ms=4, label="simulated abscissae")
ax.plot(t_mjd - EPOCH_REF, model_al, "x", ms=4, label="model at the truth")
ax.set_xlabel("days from reference epoch"); ax.set_ylabel("along-scan (mas)"); ax.legend()
plt.show()

## Simulating your own data

`OrbitSimulator` holds a truth and simulates both channels on any cadence object;
`DemoParallaxConsistentCadence` builds epochs, scan angles and the parallax factors
they imply for a sky position, through `along_scan_parallax_factor` (Gaia at L2,
astropy's built-in ephemeris — no download). Because the same factors generate and
fit the data, closure is exact by construction.

In [ ]:
from orblet.simulate.bundles import OrbitSimulator, DemoParallaxConsistentCadence
from orblet import along_scan_parallax_factor

sim = OrbitSimulator.toy_orbit(m2_msun=3.0)          # override any field of the truth
cad = DemoParallaxConsistentCadence(ra_deg=sim.ra_deg, dec_deg=sim.dec_deg, seed=3)
astro_sim = sim.simulate_astrometry(cad, seed=3)
rv_sim = sim.simulate_rv(cad, seed=3)
print("astrometric epochs:", astro_sim["obs_time"].size, " RV epochs:", np.asarray(rv_sim["obs_time_rv"]).size)

t_tcb = np.asarray(cad.astro_obs_time, dtype=float) + MJD_J2010_TCB   # J2010-days -> TCB MJD
psi_c = np.asarray(cad.astro_scan_angle, dtype=float)
pf_l2 = along_scan_parallax_factor(t_tcb, psi_c, sim.ra_deg, sim.dec_deg)
np.testing.assert_array_equal(pf_l2, cad.astro_parallax_factor_al)
print("the cadence's parallax factors ARE along_scan_parallax_factor at its epochs (identical)")

## Parallax factors: L2 versus the geocentre

For real Gaia data you never compute the parallax factor — the archive ships it
per transit. These helpers exist for simulation, for decomposing a fitted parallax
into its two sky components, and for pedagogy. The default observer is Gaia at L2:
measured against Gaia's own factor on the public BH3 bundle it agrees to 0.18 %,
while `observer="geocentre"` is 1 % low. The default ephemeris is astropy's built-in
one; a JPL kernel is `ephemeris="de432s"` after `pip install "orblet[ephemeris]"`
and a one-time `orblet.parallax.fetch_ephemeris("de432s")`.

In [ ]:
from orblet import per_direction_parallax_factors, OBSERVER_GEOCENTRE, L2_OFFSET_AU

f_a, f_d = per_direction_parallax_factors(t_tcb, sim.ra_deg, sim.dec_deg)            # L2 (default)
pf_geo = along_scan_parallax_factor(t_tcb, psi_c, sim.ra_deg, sim.dec_deg, observer=OBSERVER_GEOCENTRE)
np.testing.assert_allclose(f_a * np.sin(psi_c) + f_d * np.cos(psi_c), pf_l2)   # the projection
print(f"L2 sits {L2_OFFSET_AU:.5f} AU beyond the geocentre; "
      f"rms |L2 - geocentre| / rms(geocentre) = {np.std(pf_l2 - pf_geo) / np.std(pf_geo):.4f}")
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(cad.astro_obs_time, pf_l2, ".", label="L2 (default; the demo cadence)")
ax.plot(cad.astro_obs_time, pf_geo, "x", label="geocentre")
ax.set_xlabel("days from J2010"); ax.set_ylabel("along-scan parallax factor"); ax.legend()
plt.show()

## What these models do NOT include

- a luminous companion: every astrometric forward here takes the primary's
  orbit as the photocentre (β = 0); a luminous companion shrinks the photocentre
  orbit and biases the mass ratio low;
- correlated or per-CCD noise: the simulator adds white Gaussian noise per
  epoch and the likelihoods treat epochs as independent;
- Gaia's true observer position: the ephemeris helper is geocentric by default
  and does not model the Lissajous orbit about L2 in either mode.

Next: `02_design_matrices_and_linear_solves.ipynb` — at a fixed orbit shape the
amplitudes are linear, so most of a fit is a least-squares solve.